# Lab | Multi-modal output agent

**Create another version of the prompts below to query the dall-e vision AI model following the example shown in this notebook.**

# Multi-modal outputs: Image & Text

This notebook shows how non-text producing tools can be used to create multi-modal agents.

This example is limited to text and image outputs.

In [ ]:
!pip install langchain==0.3.25 langchain_openai==0.3.19 langchain_community==0.3.24 openai==1.109.1 pillow requests

In [ ]:
from langchain.agents import AgentType, initialize_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool

In [ ]:
import os

# Set your API key here (or set the OPENAI_API_KEY env variable before running)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_KEY_HERE")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## Dall-E

We define a `generate_image` tool that calls the DALL·E 3 API and returns the image URL.
The LangChain agent decides when to invoke the tool based on the user prompt.

In [ ]:
import openai
import sys

# Detect runtime environment
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

print(f"Running in {'Google Colab' if is_colab else 'local/Jupyter'} environment")

In [ ]:
# Global variable to capture the latest image URL so we can display it after the agent run
latest_image_url = None

@tool
def generate_image(prompt: str) -> str:
    """Generate an image using DALL·E 3 based on the given text prompt.
    Returns the URL of the generated image."""
    global latest_image_url
    client = openai.OpenAI()  # Uses OPENAI_API_KEY from environment
    response = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        n=1,
        size="1024x1024"
    )
    image_url = response.data[0].url
    latest_image_url = image_url
    return image_url

In [ ]:
# --- Helper function to display an image from URL ---
def display_image(url: str):
    """Display an image from a URL, works in both Colab and local Jupyter."""
    if url is None:
        print("No image URL available.")
        return
    if is_colab:
        from IPython.display import Image, display
        display(Image(url=url))
    else:
        from IPython.display import Image, display
        try:
            display(Image(url=url))
        except Exception:
            # Fallback: open with PIL
            from PIL import Image as PILImage
            import requests
            from io import BytesIO
            resp = requests.get(url)
            img = PILImage.open(BytesIO(resp.content))
            img.show()

In [ ]:
# Initialize the LangChain agent with the DALL·E tool
tools = [generate_image]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

mrkl = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [ ]:
# --- Original example prompt ---
result = mrkl.invoke("How would you visualize a parrot playing soccer?")
print("\nAgent response:", result["output"])

In [ ]:
# Display the generated image
display_image(latest_image_url)

---
## Assignment — Additional DALL·E Prompts

Below are additional versions of the prompt that query the DALL·E 3 model through the same agent.
Each prompt uses a different creative direction to demonstrate the model's versatility.

In [ ]:
# Assignment prompt 1: Photorealistic style
result2 = mrkl.invoke(
    "Create a photorealistic image of an astronaut riding a bicycle on the surface of Mars "
    "with Earth visible in the background."
)
print("\nAgent response:", result2["output"])

In [ ]:
display_image(latest_image_url)

In [ ]:
# Assignment prompt 2: Oil-painting / artistic style
result3 = mrkl.invoke(
    "Generate an oil-painting style image of a robot reading a book in a cozy library "
    "surrounded by shelves full of glowing books, warm candlelight."
)
print("\nAgent response:", result3["output"])

In [ ]:
display_image(latest_image_url)

In [ ]:
# Assignment prompt 3: Anime / cartoon style
result4 = mrkl.invoke(
    "Visualize a cat wearing a business suit giving a TED Talk to an audience of dogs, "
    "in a vibrant anime cartoon style."
)
print("\nAgent response:", result4["output"])

In [ ]:
display_image(latest_image_url)

---
## StableDiffusion (local GPU required)

> **Note:** This section requires a CUDA-capable GPU and enough disk space to download the model (~4 GB).  
> If running on CPU-only hardware it will be very slow. The cells below gracefully fall back to CPU.

Install the extra dependency if needed:
```
!pip install diffusers transformers accelerate torch
```

In [ ]:
try:
    import torch
    from diffusers import StableDiffusionPipeline
    import matplotlib.pyplot as plt

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if device == "cuda" else torch.float32
    print(f"Using device: {device}")

    model_id = "runwayml/stable-diffusion-v1-5"
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype)
    pipe = pipe.to(device)

    prompt = "A parrot playing soccer on a sunny day, digital art"
    image = pipe(prompt).images[0]

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title("StableDiffusion v1-5")
    plt.show()
except ImportError:
    print("diffusers / torch not installed. "
          "Run: pip install diffusers transformers accelerate torch")
except Exception as e:
    print(f"Could not run StableDiffusion pipeline: {e}")

---
## AutoPipelineForText2Image — SDXL-Turbo (local GPU required)

> **Note:** SDXL-Turbo requires a CUDA GPU with ≥8 GB VRAM and ~7 GB of disk for the model weights.  
> Same graceful fallback to CPU is applied below.

In [ ]:
try:
    import torch
    from diffusers import AutoPipelineForText2Image
    import matplotlib.pyplot as plt

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if device == "cuda" else torch.float32
    print(f"Using device: {device}")

    pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sdxl-turbo",
        torch_dtype=dtype
    ).to(device)

    prompt = "A parrot playing soccer on a sunny day, digital art"
    image = pipe(prompt, num_inference_steps=1, guidance_scale=0.0).images[0]

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title("SDXL-Turbo")
    plt.show()
except ImportError:
    print("diffusers / torch not installed. "
          "Run: pip install diffusers transformers accelerate torch")
except Exception as e:
    print(f"Could not run SDXL-Turbo pipeline: {e}")